# Week 7 v2 — Notebook 3: evaluate every detector on every set

**What it does:** reads the dataset `PATH_TO_W7_COMMON_SETS_DATASET` (manifest + masks) and every detector's result maps it can find,
then makes the SAME tables and figures for each detector (ADCD-Net, DTD, and each DAF weight), a short combined section,
and a hand-off pack. **Detectors that are not there yet are skipped with a printed note.**

**How to run:** attach `PATH_TO_W7_COMMON_SETS_DATASET` and one results dataset per detector (any that exist), Accelerator = None,
then Save & Run All. Nothing is typed in; every input is found automatically. Settings are in section 1.

---
## The research question and the project (from the Proposed Research Question document)

**Question:** How well do pre-trained document-tampering detectors generalise to diffusion-based text edits on bank
statements, and how does the edit itself (its type, size and quality), the image conditions (resizing and compression) and
the detector's training data affect their performance?

**The project in one paragraph:** each of us changes one word or number on real bank-statement pages using an AI text
editor: TextCtrl (100 pages), and AnyText and DiffSTE (the same 49 pages, edited in the same spots with the same new text).
For every one of those pages we then make three more fakes in exactly the same spot, using the three older methods that
two of our detectors (ADCD-Net and DTD) learned from: typing new text over the erased word (generation), copying text from
elsewhere on the same page (copy-move), and copying text from a different page (splicing). Each page therefore exists as
five versions, (1) AI edit, (2) generation, (3) copy-move, (4) splicing and (5) untouched, and each version is saved twice:
once as a perfect-quality PNG and once as a JPEG saved twice at quality 90. Three detectors then check every version:
ADCD-Net, DTD and DAF, with DAF run in three versions, each trained on a different AI editor (SRNet, AnyText or DiffSTE).
Every detector checks each page (1) shrunk to 512 pixels on the long side and (2) at full size in pieces, and DAF also (3)
at its own default size. Everyone scores the results with the same program (this notebook) and puts them in one table.

---
## How to read the results (written BEFORE any numbers)

- **Score of one image:** a map the same size as the page, value = 255 × probability of "tampered". A pixel is
  *flagged* when its probability is ≥ 0.5 (authors' default threshold).
- **Answer key:** one white rectangle per page (the edit's box). Every fake has exactly ONE edit, inside that box.
- **F1 (per page, then averaged):** precision = flagged pixels inside the box ÷ all flagged pixels; recall = flagged pixels
  inside the box ÷ box pixels; F1 = 2 · precision · recall ÷ (precision + recall). Pages are averaged, so a big page does
  not count more than a small one.
- **Detection rate:** share of fake pages with at least one flagged pixel inside the box.
- **False-alarm rate:** share of UNTOUCHED pages with at least one flagged pixel anywhere.
- **Why three more numbers (added after seeing the detectors' maps):** a single stray pixel is enough for the false-alarm
  rate above, so it can come close to 100% even for a decent detector. We therefore also report:
  **(1) area-based false alarm** = flagged pixels ÷ all pixels of an UNTOUCHED page, averaged over pages (lower = quieter;
  *tiny example:* 1,000 flagged of 2,000,000 pixels = 0.05%); **(2) flagged area outside the box** for edited pages (the
  same share, counted only outside the box); **(3) image-level AUC** = take one score per page (the highest map value, or
  the flagged-pixel share) and ask: pick a random edited page and a random untouched page, how often does the edited one
  score higher (ties count half)? 0.5 = a coin flip, 1 = perfect. *Tiny example:* edited scores [0.9, 0.4], untouched
  [0.5, 0.2] → pairs 0.9>0.5 yes, 0.9>0.2 yes, 0.4>0.5 no, 0.4>0.2 yes → 3 of 4 = 0.75. Edited and untouched pages are
  resampled separately for the 95% range.
- **Pixel AUC:** chance that a random pixel inside the box scores higher than a random clean pixel (1% sample of the
  clean pixels of each page). Computed from 256-bin histograms (exact, because the maps are 8-bit).
- **95% range:** 1,000 bootstrap resamples of the pages (seed 42), the 2.5th and 97.5th percentiles.
- **Contrast** = difference between two conditions, page by page (paired) where the same pages appear in both. A
  difference **counts only when its 95% range excludes 0**. Otherwise we say the data show no clear difference.
- **Wording:** we say "suggests" and "is consistent with", never "proves": one document type, a small number of pages,
  and the detectors' training data are not fully known to us.
- **What each contrast answers:** *edit type* (generation, copy-move, splicing versus the AI edit), *edit size* (large
  versus small boxes), *edit quality* (AI edits OCR read correctly versus not), *resizing* (`full` versus `shrink512`;
  `default` versus `full`, DAF only), *compression* (JPEG versus PNG), *training data* (DAF weights trained on the same
  editor versus the other editor).
- **Caution:** TextCtrl edits are letters and digits only; AnyText and DiffSTE edits also contain punctuation, amounts and
  dates. AnyText and DiffSTE share the same 49 pages, boxes, new text and non-AI versions.

## 1. CONFIG

**Why (ELI5):** every setting in one place. The notebook is meant to run with no changes, so the defaults are the real
run (`SMOKE=False`). `SMOKE=True` scores only 3 pages per set, for a quick test.

In [ ]:
# standard tools for files, text, time and random numbers
import os, re, io, sys, json, glob, time, zipfile, shutil, pickle, zlib, warnings
# the date for the report
from datetime import datetime
# arrays and tables
import numpy as np
import pandas as pd
# reading and resizing images
from PIL import Image
# plotting without a screen
import matplotlib
# draw into files
matplotlib.use("Agg")
# the plotting functions
import matplotlib.pyplot as plt

# ======================= CONFIG: change settings here and nowhere else =======================
# False = the real run; True = only SMOKE_N pages per set (quick test)
SMOKE = False
# pages per set in a smoke run
SMOKE_N = 3
# seed for every random choice
SEED = 42
# bootstrap resamples for the 95% ranges
N_BOOT = 1000
# a pixel is flagged when its probability is at least this
THR = 0.5
# the same threshold on the 8-bit scale: value >= 128 means 128/255 = 0.502 >= 0.5 (127/255 = 0.498 is below)
THR_VALUE = 128
# share of clean pixels sampled per page for the pixel AUC
NEG_FRAC = 0.01
# tiny number that avoids dividing by zero (as in Week 7 notebook 07)
EPS = 1e-8
# False = STOP if a detector's results miss any page; True = score what exists and note the gaps
ALLOW_INCOMPLETE = False
# optional: type one folder that holds ALL inputs (leave "" normally)
INPUT_ROOT_OVERRIDE = ""
# =============================================================================================

# the detector runs we expect (only the ones that exist are used)
EXPECTED_RUNS = ["adcd", "dtd", "daf_srnet", "daf_anytext", "daf_diffste"]
# the sizes each detector family is run at
EXPECTED_SIZES = {"adcd": ["shrink512", "full"], "dtd": ["shrink512", "full"], "daf": ["shrink512", "full", "default"]}
# all size names in a fixed order
SIZES_ALL = ["shrink512", "full", "default"]
# the five versions and the four that contain an edit
VERSIONS = ["ai_edit", "generation", "copy_move", "splicing", "untouched"]
# only these have a tampered box
EDITED = ["ai_edit", "generation", "copy_move", "splicing"]
# the two file formats
FORMATS = ["png", "jpg"]
# the three page sets
SETS = ["textctrl", "anytext", "diffste"]

# where are we running?
IN_KAGGLE = os.path.exists("/kaggle/input")
# Colab check
IN_COLAB = "google.colab" in sys.modules
# Kaggle
if IN_KAGGLE:
    # where inputs are mounted
    SEARCH_ROOTS = ["/kaggle/input"]
    # where outputs go
    WORKING = "/kaggle/working"
# Colab
elif IN_COLAB:
    # look in /content and Drive
    SEARCH_ROOTS = ["/content", "/content/drive/MyDrive"]
    # outputs
    WORKING = "/content/w7_eval_working"
# laptop
else:
    # look in the current folder
    SEARCH_ROOTS = [os.getcwd()]
    # outputs
    WORKING = os.path.join(os.getcwd(), "w7_eval_working")
# an explicit input folder wins
if INPUT_ROOT_OVERRIDE:
    # use it
    SEARCH_ROOTS = [INPUT_ROOT_OVERRIDE]
# the output folder that gets zipped
EVAL_DIR = os.path.join(WORKING, "eval_outputs")
# scratch folder (cache), not zipped
WORK_DIR = os.path.join(WORKING, "w7_eval_work")
# make the folders
for d in [EVAL_DIR, WORK_DIR]:
    # create it
    os.makedirs(d, exist_ok=True)
# seeds
np.random.seed(SEED)
# save the settings
config = dict(SMOKE=SMOKE, SMOKE_N=SMOKE_N, SEED=SEED, N_BOOT=N_BOOT, THR=THR, THR_VALUE=THR_VALUE, NEG_FRAC=NEG_FRAC,
              ALLOW_INCOMPLETE=ALLOW_INCOMPLETE, started=datetime.now().isoformat(timespec="seconds"))
# write config.json
with open(os.path.join(EVAL_DIR, "config.json"), "w") as f:
    json.dump(config, f, indent=2)
# the list of checks (name, value, passed, hard)
CHECKS = []

# record one check and print it
def check(name, value, passed, hard=True):
    # drop an older entry with the same name (a re-run cell must replace its result, not keep the old FAIL)
    CHECKS[:] = [c for c in CHECKS if c["check"] != name]
    # store it
    CHECKS.append(dict(check=name, value=value, passed=bool(passed), hard=hard))
    # show it
    print(("PASS " if passed else ("FAIL " if hard else "WARN ")) + name + " -> " + str(value))

# stop with a clear message if any hard check failed
def stop_if_failed():
    # the hard failures
    failed = [c["check"] for c in CHECKS if c["hard"] and not c["passed"]]
    # stop
    if failed:
        raise RuntimeError("STOP: these checks failed: " + "; ".join(failed))

# big warning in smoke mode
if SMOKE:
    print("#" * 78)
    print("#  SMOKE=True -> only", SMOKE_N, "pages per set. Set SMOKE=False before Save & Run All.")
    print("#" * 78)
# show the settings
print(config)

## 2. Find the inputs (no typing)

**Why (ELI5):** Kaggle puts each attached dataset in its own folder, sometimes nested. We look for three things by their
marker files: `manifest.csv` (the dataset), `run_info.json` (one folder per detector run) and `decisions.md` (the notes,
optional). Missing detectors are not an error: they are listed and skipped.

In [ ]:
# folders we never walk into (they hold thousands of images or our own output)
SKIP_DIRS = {"images", "masks", "shrink512", "full", "default", "figures", "cache", "eval_outputs", "w7_eval_working", "w7_eval_work"}

# list every folder (up to max_depth levels down) that contains a file called `marker`
def find_folders_with(marker, max_depth=6):
    # the answer
    found = []
    # look under each search root
    for root in SEARCH_ROOTS:
        # skip roots that do not exist
        if not os.path.isdir(root):
            continue
        # walk down from the root
        for folder, subfolders, files in os.walk(root, followlinks=True):
            # how many levels below the root
            depth = folder[len(root):].count(os.sep)
            # the marker is here
            if marker in files:
                found.append(folder)
            # do not enter big folders or go too deep
            subfolders[:] = [] if depth >= max_depth else [d for d in subfolders if d not in SKIP_DIRS]
    # give back the folders
    return found

# ---------- the dataset (manifest) ----------
# every folder with a manifest.csv
cands = find_folders_with("manifest.csv")
# nothing found: stop with a clear message
if not cands:
    raise FileNotFoundError("No manifest.csv found. Attach the dataset PATH_TO_W7_COMMON_SETS_DATASET. Looked under: " + str(SEARCH_ROOTS))
# prefer the folder that also holds masks/ (the real dataset)
cands.sort(key=lambda p: (not os.path.isdir(os.path.join(p, "masks")), p))
# the dataset folder
SETS_DIR = cands[0]
# read the manifest, text columns as text
manifest = pd.read_csv(os.path.join(SETS_DIR, "manifest.csv"), dtype=str, keep_default_na=False)
# numeric columns
for c in ["x1", "y1", "x2", "y2"]:
    # to integers
    manifest[c] = manifest[c].astype(int)
# box share and OCR numbers as numbers
manifest["box_share"] = manifest["box_share"].astype(float)
# OCR correctness: True / False / unknown (blank for non-AI rows)
manifest["ai_ocr_correct"] = manifest["ai_ocr_correct"].map({"True": True, "False": False})
# SMOKE: keep the first SMOKE_N pages of each set
if SMOKE:
    # pages to keep
    keep = {s: sorted(manifest[manifest["set"] == s]["page"].unique())[:SMOKE_N] for s in SETS}
    # filter
    manifest = manifest[[p in keep[s] for s, p in zip(manifest["set"], manifest["page"])]].reset_index(drop=True)
# one row per unique image file (the anytext/diffste rows share the shared49 files)
images_u = manifest.drop_duplicates("path").reset_index(drop=True)
# split the path: images/<image set>/<version>/<format>/<page>.<ext>
parts = images_u["path"].str.split("/", expand=True)
# the folder set the image lives in (textctrl, shared49, anytext, diffste)
images_u["img_set"] = parts[1]
# file stem (the page id)
images_u["stem"] = parts[4].str.rsplit(".", n=1).str[0]
# print what we found
print("dataset folder :", SETS_DIR)
print("manifest rows  :", len(manifest), "| unique image files:", len(images_u))
print(manifest.groupby(["set", "version"]).size().unstack(fill_value=0).to_string())

# ---------- the detector runs (run_info.json) ----------
# run id -> dict(folder, detector, weights, sizes)
RUNS = {}
# every folder with a run_info.json
for folder in find_folders_with("run_info.json"):
    # read its info
    with open(os.path.join(folder, "run_info.json")) as f:
        # the dictionary
        info = json.load(f)
    # the run id = the folder name (e.g. adcd, dtd, daf_anytext)
    rid = os.path.basename(folder.rstrip("/"))
    # the detector family and the weights name
    family = str(info.get("detector", rid.split("_")[0])).lower()
    # DAF weights: from run_info or from the folder name
    weights = str(info.get("weights", "")) if isinstance(info.get("weights", ""), str) else ""
    # a plain "daf" folder with weights in run_info becomes daf_<weights>
    if rid == "daf" and weights:
        rid = "daf_" + weights
    # weights from the folder name when missing
    if family == "daf" and not weights and "_" in rid:
        weights = rid.split("_", 1)[1]
    # the folder name is the authority for the detector and the weights (a run_info.json with other key names must not matter)
    # the weight word anywhere in the folder name (e.g. daf-results-srnet or daf_srnet)
    wword = next((w for w in ["srnet", "anytext", "diffste"] if w in rid.lower()), None)
    # a DAF folder named after a weight: the run id becomes daf_<weight>
    if wword and "daf" in rid.lower():
        # DAF with that weight
        family, weights, rid = "daf", wword, "daf_" + wword
    # DAF with the weights named in the folder (daf_<weight> style)
    elif rid.startswith("daf_"):
        # DAF with the weights named in the folder
        family, weights = "daf", rid.split("_", 1)[1]
    # ADCD-Net and DTD
    elif rid in ("adcd", "dtd"):
        # family = folder name, no weights name
        family, weights = rid, ""
    # sizes that exist as folders
    sizes = [s for s in SIZES_ALL if os.path.isdir(os.path.join(folder, s))]
    # do not register the same run twice
    if rid in RUNS:
        print("NOTE: run", rid, "found twice; using", RUNS[rid]["folder"])
        continue
    # keep it
    RUNS[rid] = dict(folder=folder, detector=family, weights=weights, sizes=sizes)
# fixed order: ADCD-Net, DTD, DAF weights, then anything else
RUNS = {k: RUNS[k] for k in sorted(RUNS, key=lambda r: EXPECTED_RUNS.index(r) if r in EXPECTED_RUNS else 99)}
# report what exists and what does not
for rid in EXPECTED_RUNS:
    # found
    if rid in RUNS:
        print("FOUND    ", rid, "sizes:", RUNS[rid]["sizes"], "|", RUNS[rid]["folder"])
    # not there yet
    else:
        print("NOT FOUND", rid, "-> skipped (attach its results dataset and run again)")
# runs that are not in the expected list are still used
for rid in RUNS:
    # extra run
    if rid not in EXPECTED_RUNS:
        print("EXTRA    ", rid, "sizes:", RUNS[rid]["sizes"], "|", RUNS[rid]["folder"])
# without any detector there is nothing to evaluate
if not RUNS:
    raise FileNotFoundError("No detector results found (a folder with run_info.json). Attach at least one results dataset.")

# ---------- were all runs made from THIS version of PATH_TO_W7_COMMON_SETS_DATASET? ----------
# standard tool for the fingerprint
import hashlib
# md5 of this dataset's manifest.csv
MANIFEST_MD5 = hashlib.md5(open(os.path.join(SETS_DIR, "manifest.csv"), "rb").read()).hexdigest()
# one row per run
md5_rows = []
# each run
for rid in RUNS:
    # read its run_info.json
    with open(os.path.join(RUNS[rid]["folder"], "run_info.json")) as f:
        # the dictionary
        ri = json.load(f)
    # the recorded fingerprint (empty if the run did not record one)
    rec = str(ri.get("manifest_md5", ""))
    # same, different or unknown
    md5_rows.append(dict(run=rid, recorded=(rec[:8] if rec else "(not recorded)"), same_as_this_dataset=("yes" if rec == MANIFEST_MD5 else ("no" if rec else "unknown"))))
# show the table
print("manifest fingerprint of this dataset:", MANIFEST_MD5[:8])
print(pd.DataFrame(md5_rows).to_string(index=False))
# a run made from a different manifest would have different pages: stop
wrong = [r["run"] for r in md5_rows if r["same_as_this_dataset"] == "no"]
# hard check
check("every run was made from this version of PATH_TO_W7_COMMON_SETS_DATASET", "different: " + str(wrong) + " | not recorded: " + str([r["run"] for r in md5_rows if r["same_as_this_dataset"] == "unknown"]), len(wrong) == 0)
# stop now with a clear message
if wrong:
    # message
    raise RuntimeError("STOP: these runs were made from a DIFFERENT manifest.csv than the PATH_TO_W7_COMMON_SETS_DATASET attached here: " + str(wrong) + ". Attach the same dataset version that was used for the runs (or rerun them).")

# ---------- the notes (optional) ----------
# decisions.md anywhere in the inputs
notes_found = find_folders_with("decisions.md")
# copy the first one next to the outputs
DECISIONS_TEXT = ""
# found
if notes_found:
    # read it
    with open(os.path.join(notes_found[0], "decisions.md")) as f:
        # keep the text
        DECISIONS_TEXT = f.read()
    # copy it into the outputs
    shutil.copy(os.path.join(notes_found[0], "decisions.md"), os.path.join(EVAL_DIR, "decisions_copy.md"))
    # say so
    print("decisions.md found:", notes_found[0], "(", len(DECISIONS_TEXT.splitlines()), "lines )")
# not attached
else:
    # say so
    print("decisions.md not attached (optional)")

## 3. Page-count check (stops early, before any slow work)

**Why (ELI5):** if a detector's folder is missing some pages, every average would silently use different pages. So we
count, for every detector and size, how many images the manifest lists and how many result files exist. Any gap stops
the notebook with a clear message (set `ALLOW_INCOMPLETE=True` in section 1 to score what exists anyway).

Where a result file lives: `<run>/<size>/<image set>/<version>/<format>/<page>.png`, with the image set exactly as in the
image path (`textctrl`, `shared49`, `anytext` or `diffste`).

In [ ]:
# the path of a result map for one image
def result_path(run, size, row):
    # <run folder>/<size>/<image set>/<version>/<format>/<page>.png
    return os.path.join(RUNS[run]["folder"], size, row["img_set"], row["version"], row["format"], row["stem"] + ".png")

# one row per run and size: expected, found, missing
cov_rows = []
# every run
for run, info in RUNS.items():
    # the sizes this detector family should have (plus any extra size folders it has)
    wanted = EXPECTED_SIZES.get(info["detector"], SIZES_ALL)
    # the sizes to check
    for size in sorted(set(wanted) | set(info["sizes"]), key=SIZES_ALL.index):
        # which images exist
        have = [os.path.exists(result_path(run, size, r)) for r in images_u.to_dict("records")]
        # one row
        cov_rows.append(dict(run=run, size=size, expected=len(images_u), found=int(sum(have)), missing=int(len(have) - sum(have))))
# the coverage table
coverage = pd.DataFrame(cov_rows)
# save it
coverage.to_csv(os.path.join(EVAL_DIR, "coverage.csv"), index=False)
# show it
print(coverage.to_string(index=False))
# runs and sizes with gaps
gaps = coverage[coverage["missing"] > 0]
# a hard check unless gaps are allowed
check("every detector has every page at every size", str(len(gaps)) + " incomplete " + str(gaps[["run", "size", "missing"]].values.tolist()[:6]),
      len(gaps) == 0, hard=not ALLOW_INCOMPLETE)
# stop here with an explanation
if len(gaps) and not ALLOW_INCOMPLETE:
    # the message
    raise RuntimeError("STOP: some detector results are missing pages (see the table above). Finish those runs, or set ALLOW_INCOMPLETE=True in section 1 to score what exists.")

## 4. Score every image (resumable)

**Why (ELI5):** for each result map we ask: how many flagged pixels are inside the box, how many outside, and what is the
highest score? One map = one row of numbers. The slow part is reading thousands of maps, so each detector-and-size is
saved when finished and skipped on a re-run.

**Formulas (box = answer key, flagged = value ≥ 128 of 255):**
precision = TP ÷ (flagged); recall = TP ÷ (box pixels); F1 = 2PR ÷ (P + R); IoU = TP ÷ (flagged + box − TP), where TP =
flagged pixels inside the box. *Tiny example:* box 100 px, flagged 80 px of which 60 inside → P = 0.75, R = 0.60,
F1 = 0.667.

`untouched` pages have no edit, so only `max_value`, `flagged_any` and `mean_in_box` are filled for them.

In [ ]:
# one mask at a time is kept in memory (images are processed page by page)
_mask_cache = {}

# read a mask and make the repeatable 1% clean-pixel sample for this page
def get_mask(mask_rel):
    # already loaded
    if _mask_cache.get("key") == mask_rel:
        # reuse it
        return _mask_cache["val"]
    # read the mask: white = box
    box = np.asarray(Image.open(os.path.join(SETS_DIR, mask_rel)).convert("L")) > 127
    # how many clean pixels there are
    n_clean = int((~box).sum())
    # how many to sample
    k = int(round(NEG_FRAC * n_clean))
    # generator seeded by (seed, page): the same pixels in every version and every detector
    rng = np.random.default_rng([SEED, zlib.crc32(mask_rel.encode())])
    # positions (inside the list of clean pixels) of the sample
    idx = np.sort(rng.choice(n_clean, size=k, replace=False)) if k > 0 else np.zeros(0, dtype=np.int64)
    # remember
    _mask_cache["key"] = mask_rel
    # the mask and the sample
    _mask_cache["val"] = (box, idx)
    # give back
    return _mask_cache["val"]

# read a result map as 8-bit numbers at the page size
def read_map(path, shape):
    # open as greyscale
    im = Image.open(path).convert("L")
    # the wanted size is (width, height)
    if im.size != (shape[1], shape[0]):
        # resize with bilinear interpolation (should not be needed)
        im = im.resize((shape[1], shape[0]), Image.BILINEAR)
        # note it
        resized = True
    # same size
    else:
        # no resize
        resized = False
    # to an array
    return np.asarray(im), resized

# score one result map against its mask
def score_map(arr, box, idx, edited):
    # flagged pixels
    pred = arr >= THR_VALUE
    # how many flagged pixels, how many in the box
    pred_n, gt_n = int(pred.sum()), int(box.sum())
    # flagged pixels inside the box
    tp = int((pred & box).sum())
    # the numbers every image gets
    out = dict(max_value=float(arr.max()) / 255.0, flagged_any=float(pred_n > 0), mean_in_box=float(arr[box].mean()) / 255.0)
    # share of ALL pixels that are flagged, and share of the pixels OUTSIDE the box that are flagged (area-based, not "any pixel")
    out.update(flagged_frac=pred_n / pred.size, flagged_frac_outside=int((pred & ~box).sum()) / max(1, int((~box).sum())))
    # the numbers only edited pages get
    if edited:
        # precision, recall, F1, IoU
        p = tp / (pred_n + EPS)
        # recall
        r = tp / (gt_n + EPS)
        # F1
        f1 = 2 * p * r / (p + r + EPS)
        # IoU
        iou = tp / (pred_n + gt_n - tp + EPS)
        # store them
        out.update(f1=f1, precision=p, recall=r, iou=iou, detected=float(tp > 0))
    # untouched pages: not defined
    else:
        # NaN
        out.update(f1=np.nan, precision=np.nan, recall=np.nan, iou=np.nan, detected=np.nan)
    # histograms for the pixel AUC (edited pages only): tampered = box pixels, clean = the 1% sample
    if edited:
        # 256 counts of the values inside the box
        pos = np.bincount(arr[box], minlength=256).astype(np.int64)
        # 256 counts of the sampled clean values
        neg = np.bincount(arr[~box][idx], minlength=256).astype(np.int64)
    # untouched pages have no tampered pixels
    else:
        # nothing
        pos, neg = None, None
    # give back
    return out, pos, neg

# cache folder (separate for smoke runs)
CACHE_DIR = os.path.join(WORK_DIR, "cache2_smoke" if SMOKE else "cache2")
# create it
os.makedirs(CACHE_DIR, exist_ok=True)
# all per-image rows and histograms
img_rows = []
# (run, size, path) -> (pos histogram, neg histogram)
HIST = {}
# images whose map had to be resized
n_resized = 0
# stopwatch
t_score = time.time()
# every run and size
for run, info in RUNS.items():
    # the sizes of this run
    for size in sorted(set(info["sizes"]), key=SIZES_ALL.index):
        # cache file of this run and size
        cache = os.path.join(CACHE_DIR, run + "__" + size + ".pkl")
        # finished earlier: load it
        if os.path.exists(cache):
            # read
            with open(cache, "rb") as f:
                # rows and histograms
                rows_c, hist_c = pickle.load(f)
        # not finished: score it
        else:
            # rows and histograms of this run and size
            rows_c, hist_c = [], {}
            # images sorted by mask so one mask serves all versions of a page
            for r in images_u.sort_values(["mask_path", "path"]).to_dict("records"):
                # the result file
                p = result_path(run, size, r)
                # missing (only possible with ALLOW_INCOMPLETE): skip
                if not os.path.exists(p):
                    continue
                # the mask and the clean sample
                box, idx = get_mask(r["mask_path"])
                # the map
                arr, was_resized = read_map(p, box.shape)
                # count resized maps
                n_resized += int(was_resized)
                # the numbers
                out, pos, neg = score_map(arr, box, idx, r["version"] != "untouched")
                # one row
                rows_c.append(dict(run=run, size=size, path=r["path"], **out))
                # keep the histograms
                if pos is not None:
                    # store
                    hist_c[r["path"]] = (pos, neg)
            # save the cache
            with open(cache, "wb") as f:
                # write
                pickle.dump((rows_c, hist_c), f)
        # collect
        img_rows += rows_c
        # histograms with the full key
        for path, h in hist_c.items():
            # key
            HIST[(run, size, path)] = h
        # progress
        print("scored", run, size, len(rows_c), "images")
# one table of unique images x runs x sizes
img = pd.DataFrame(img_rows)
# expand to the manifest rows (anytext/diffste rows share the shared49 images)
per_page_full = manifest.merge(img, on="path", how="inner")
# add the detector family and weights
per_page_full["detector"] = [RUNS[r]["detector"] for r in per_page_full["run"]]
# weights
per_page_full["weights"] = [RUNS[r]["weights"] for r in per_page_full["run"]]
# summary line
print("per-image rows:", len(img), "| per-page rows (manifest sets):", len(per_page_full), "| maps resized:", n_resized, "|", int(time.time() - t_score), "s")
# a hard check: the number of rows equals manifest rows x run-sizes that exist
n_expected = len(manifest) * sum(len(i["sizes"]) for i in RUNS.values())
# compare
check("per-page rows = manifest rows x detector sizes", str(len(per_page_full)) + " vs " + str(n_expected), len(per_page_full) == n_expected, hard=not ALLOW_INCOMPLETE)
# maps of the wrong size are a warning
check("result maps that had to be resized to the page size", str(n_resized), n_resized == 0, hard=False)
# stop if needed
stop_if_failed()

## 5. Tables per detector: `per_page.csv`, `summary.csv`, `contrasts.csv`

**Why (ELI5):** now we average. For each detector, size, set, version and format we report the mean of each metric over
pages with a 95% range. Then we compare conditions page by page.

**95% bootstrap range (WHAT/WHY/HOW):** WHAT: the range the average would likely fall in if we had drawn other pages.
WHY: with 49–100 pages the average alone can mislead. HOW: draw the same number of pages with replacement 1,000 times,
average each draw, and take the 2.5th and 97.5th percentiles. *Tiny example:* pages [0, 1, 1] → draws like [1, 1, 0] →
means 0.67, 1.0, 0.33, … the middle 95% of those means is the range.

**Contrast = difference.** Paired: for each page, (condition A − condition B), then the bootstrap of the mean
difference. Unpaired (different pages in each group, e.g. large vs small boxes): bootstrap each group's mean separately
and subtract. A contrast **counts only if its range excludes 0**.

**Pixel AUC from histograms:** for tampered counts T_k and clean counts C_k in value bin k,
AUC = Σ_k T_k (Σ_{j<k} C_j + ½ C_k) ÷ (ΣT · ΣC). *Tiny example:* one tampered pixel at 160, clean pixels at 20 and 220
→ it beats 1 of 2 → AUC 0.5.

In [ ]:
# 95% bootstrap range of a mean (NaN skipped); returns mean, low, high
def boot_mean_ci(values, n_boot=N_BOOT, seed=SEED):
    # keep the defined values
    v = np.asarray(values, dtype=float)
    # drop NaN
    v = v[~np.isnan(v)]
    # nothing left
    if len(v) == 0:
        return np.nan, np.nan, np.nan
    # same seed, same range every time
    rng = np.random.default_rng(seed)
    # resample and average
    means = rng.choice(v, size=(n_boot, len(v)), replace=True).mean(axis=1)
    # the mean and the 2.5th / 97.5th percentiles
    return v.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5)

# 95% range of the difference of two UNPAIRED group means (a minus b)
def boot_unpaired(a, b, n_boot=N_BOOT, seed=SEED):
    # arrays without NaN
    a, b = np.asarray(a, float), np.asarray(b, float)
    # drop NaN
    a, b = a[~np.isnan(a)], b[~np.isnan(b)]
    # both groups need at least 2 pages
    if len(a) < 2 or len(b) < 2:
        return np.nan, np.nan, np.nan
    # seeded generator
    rng = np.random.default_rng(seed)
    # resampled means of each group
    ma = rng.choice(a, size=(n_boot, len(a)), replace=True).mean(axis=1)
    # second group
    mb = rng.choice(b, size=(n_boot, len(b)), replace=True).mean(axis=1)
    # difference of the resampled means
    d = ma - mb
    # the observed difference and its range
    return a.mean() - b.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5)

# AUC from two 256-bin count arrays (ties count half)
def auc_hist(pos_h, neg_h):
    # totals
    n_pos, n_neg = pos_h.sum(), neg_h.sum()
    # undefined without both kinds of pixels
    if n_pos == 0 or n_neg == 0:
        return np.nan
    # clean pixels in lower bins than each bin
    below = np.cumsum(neg_h) - neg_h
    # Mann-Whitney formula
    return float((pos_h * (below + 0.5 * neg_h)).sum() / (n_pos * n_neg))

# pixel AUC with a bootstrap over pages
def boot_auc_hist(pos_mat, neg_mat, n_boot=N_BOOT, seed=SEED):
    # the AUC of all pages together
    auc = auc_hist(pos_mat.sum(axis=0), neg_mat.sum(axis=0))
    # seeded generator
    rng = np.random.default_rng(seed)
    # number of pages
    n = len(pos_mat)
    # AUC of each resample
    vals = []
    # resample pages
    for _ in range(n_boot):
        # page indices
        idx = rng.integers(0, n, n)
        # pooled AUC
        a = auc_hist(pos_mat[idx].sum(axis=0), neg_mat[idx].sum(axis=0))
        # keep defined values
        if not np.isnan(a):
            # store
            vals.append(a)
    # no value: undefined
    if not vals:
        return auc, np.nan, np.nan
    # the AUC and its range
    return auc, np.percentile(vals, 2.5), np.percentile(vals, 97.5)

# the per-page metrics we summarise (column in per_page -> metric name)
PAGE_METRICS = [("f1", "f1"), ("precision", "precision"), ("recall", "recall"), ("iou", "iou"),
                ("detected", "detection_rate"), ("mean_in_box", "mean_in_box")]

# the columns of per_page.csv (the contract)
PER_PAGE_COLS = ["detector", "weights", "size", "set", "version", "format", "page", "f1", "precision", "recall", "iou", "detected",
                 "max_value", "flagged_any", "mean_in_box", "flagged_frac", "flagged_frac_outside"]

# summary rows of one run (long format)
def summarise_run(pp, run):
    # rows
    rows = []
    # detector and weights of this run
    det, wts = RUNS[run]["detector"], RUNS[run]["weights"]
    # one group = size x set x version x format
    for (size, set_, version, fmt), g in pp.groupby(["size", "set", "version", "format"]):
        # a helper that adds a row
        def add(metric, value, lo, hi, n):
            # append
            rows.append(dict(detector=det, weights=wts, size=size, set=set_, version=version, format=fmt, metric=metric,
                             value=value, ci_lo=lo, ci_hi=hi, n=int(n)))
        # untouched pages: false-alarm rate (any flagged pixel on the page) and the mean of the maximum
        if version == "untouched":
            # false-alarm rate
            m, lo, hi = boot_mean_ci(g["flagged_any"])
            # add
            add("false_alarm_rate", m, lo, hi, len(g))
            # average highest score
            m, lo, hi = boot_mean_ci(g["max_value"])
            # add
            add("max_value", m, lo, hi, len(g))
            # area-based false alarm: the share of the untouched page that is flagged
            m, lo, hi = boot_mean_ci(g["flagged_frac"])
            # add
            add("false_alarm_area", m, lo, hi, len(g))
            # next group
            continue
        # edited pages: every per-page metric
        for col, name in PAGE_METRICS:
            # mean and range
            m, lo, hi = boot_mean_ci(g[col])
            # add
            add(name, m, lo, hi, g[col].notna().sum())
        # area-based false positives: the share of the pixels OUTSIDE the box that are flagged
        m, lo, hi = boot_mean_ci(g["flagged_frac_outside"])
        # add
        add("flagged_area_outside", m, lo, hi, len(g))
        # pixel AUC from the histograms of these pages
        pos_mat = np.stack([HIST[(run, size, p)][0] for p in g["path"]])
        # clean-pixel histograms
        neg_mat = np.stack([HIST[(run, size, p)][1] for p in g["path"]])
        # AUC and range
        a, lo, hi = boot_auc_hist(pos_mat, neg_mat)
        # add
        add("pixel_auc", a, lo, hi, len(g))
    # the table
    return pd.DataFrame(rows)

# image-level AUC: can ONE number per page tell an edited page from an untouched page?
def image_auc_rows(pp, run):
    # rows
    rows = []
    # detector and weights
    det, wts = RUNS[run]["detector"], RUNS[run]["weights"]
    # one group = size x set x format
    for (size, set_, fmt), g in pp.groupby(["size", "set", "format"]):
        # the untouched pages are the negatives
        neg_g = g[g["version"] == "untouched"]
        # need a few
        if len(neg_g) < 2:
            continue
        # each edited version against the untouched pages
        for version in EDITED:
            # the positives
            pos_g = g[g["version"] == version]
            # need a few
            if len(pos_g) < 2:
                continue
            # two scores: the highest map value, and the share of flagged pixels
            for col, name in [("max_value", "image_auc_max"), ("flagged_frac", "image_auc_area")]:
                # scores
                pos, neg = pos_g[col].values.astype(float), neg_g[col].values.astype(float)
                # table of all pairs: 1 if the edited page scores higher, 0.5 for a tie, 0 if lower
                P = (pos[:, None] > neg[None, :]).astype(float) + 0.5 * (pos[:, None] == neg[None, :])
                # AUC = the average over all pairs
                auc = float(P.mean())
                # seeded generator
                rng = np.random.default_rng(SEED)
                # resampled page indices (edited and untouched pages are resampled separately)
                ip = rng.integers(0, len(pos), size=(N_BOOT, len(pos)))
                # untouched
                ineg = rng.integers(0, len(neg), size=(N_BOOT, len(neg)))
                # the AUC of every resample
                vals = np.array([P[np.ix_(ip[b], ineg[b])].mean() for b in range(N_BOOT)])
                # the row (n = number of edited pages; the untouched pages are the negatives)
                rows.append(dict(detector=det, weights=wts, size=size, set=set_, version=version, format=fmt, metric=name,
                                 value=auc, ci_lo=float(np.percentile(vals, 2.5)), ci_hi=float(np.percentile(vals, 97.5)), n=int(len(pos))))
    # the table
    return pd.DataFrame(rows)

# summary by size group (small / medium / large boxes)
def summarise_size_groups(pp, run):
    # rows
    rows = []
    # detector and weights
    det, wts = RUNS[run]["detector"], RUNS[run]["weights"]
    # edited pages only
    for (size, set_, version, fmt, sg), g in pp[pp["version"] != "untouched"].groupby(["size", "set", "version", "format", "size_group"]):
        # the two headline metrics
        for col, name in [("f1", "f1"), ("detected", "detection_rate")]:
            # mean and range
            m, lo, hi = boot_mean_ci(g[col])
            # row
            rows.append(dict(detector=det, weights=wts, size=size, set=set_, version=version, format=fmt, size_group=sg, metric=name,
                             value=m, ci_lo=lo, ci_hi=hi, n=int(len(g))))
    # the table
    return pd.DataFrame(rows)

# paired contrasts: factor A level `plus` minus level `minus`, page by page
def paired_contrast(pp, run, family, name, factor, plus, minus, index_cols, edited_only=True):
    # rows
    rows = []
    # edited versions only (untouched has no F1)
    sub = pp[pp["version"].isin(EDITED)] if edited_only else pp
    # only the two levels
    sub = sub[sub[factor].isin([plus, minus])]
    # nothing to compare
    if len(sub) == 0:
        return rows
    # both metrics
    for col, metric in [("f1", "f1"), ("detected", "detection_rate")]:
        # one row per page and context, one column per level
        wide = sub.pivot(index=index_cols + ["page"], columns=factor, values=col)
        # both levels must exist
        if plus not in wide.columns or minus not in wide.columns:
            continue
        # keep pages that have both
        wide = wide.dropna(subset=[plus, minus])
        # the paired differences
        wide["d"] = wide[plus] - wide[minus]
        # the contexts (everything in the index except the page)
        for ctx, g in wide.groupby(level=index_cols):
            # make the context a tuple
            ctx = ctx if isinstance(ctx, tuple) else (ctx,)
            # context as a dictionary
            c = dict(zip(index_cols, ctx))
            # mean difference and range
            m, lo, hi = boot_mean_ci(g["d"].values)
            # add the row
            rows.append(dict(detector=RUNS[run]["detector"], weights=RUNS[run]["weights"], contrast=name, family=family,
                             set=c.get("set", ""), size=c.get("size", ""), version=c.get("version", ""), format=c.get("format", ""),
                             metric=metric, diff=m, ci_lo=lo, ci_hi=hi, n=int(len(g)), excludes_zero=bool(lo > 0 or hi < 0)))
    # the rows
    return rows

# all contrasts of one run
def contrasts_run(pp, run):
    # rows
    rows = []
    # edit type: each DocTamper version minus the AI edit
    for v in ["generation", "copy_move", "splicing"]:
        # paired by page
        rows += paired_contrast(pp, run, "edit_type", v + " - ai_edit", "version", v, "ai_edit", ["size", "set", "format"])
    # resizing: full minus shrink512, default minus full
    rows += paired_contrast(pp, run, "resizing", "full - shrink512", "size", "full", "shrink512", ["set", "version", "format"])
    # default minus full (DAF)
    rows += paired_contrast(pp, run, "resizing", "default - full", "size", "default", "full", ["set", "version", "format"])
    # compression: jpg minus png
    rows += paired_contrast(pp, run, "compression", "jpg - png", "format", "jpg", "png", ["size", "set", "version"])
    # edit size and edit quality: unpaired
    for (size, set_, version, fmt), g in pp[pp["version"].isin(EDITED)].groupby(["size", "set", "version", "format"]):
        # large minus small boxes
        for col, metric in [("f1", "f1"), ("detected", "detection_rate")]:
            # large and small values
            a = g[g["size_group"] == "large"][col].values
            # small
            b = g[g["size_group"] == "small"][col].values
            # difference and range
            m, lo, hi = boot_unpaired(a, b)
            # skip undefined
            if not np.isnan(m):
                # add
                rows.append(dict(detector=RUNS[run]["detector"], weights=RUNS[run]["weights"], contrast="large - small", family="edit_size",
                                 set=set_, size=size, version=version, format=fmt, metric=metric, diff=m, ci_lo=lo, ci_hi=hi,
                                 n=int(len(a) + len(b)), excludes_zero=bool(lo > 0 or hi < 0)))
        # edit quality: only AI edits have an OCR result
        if version == "ai_edit":
            # both metrics
            for col, metric in [("f1", "f1"), ("detected", "detection_rate")]:
                # correct and not correct
                a = g[g["ai_ocr_correct"] == True][col].values
                # not correct
                b = g[g["ai_ocr_correct"] == False][col].values
                # difference and range
                m, lo, hi = boot_unpaired(a, b)
                # skip undefined
                if not np.isnan(m):
                    # add
                    rows.append(dict(detector=RUNS[run]["detector"], weights=RUNS[run]["weights"], contrast="ocr_correct - not", family="edit_quality",
                                     set=set_, size=size, version=version, format=fmt, metric=metric, diff=m, ci_lo=lo, ci_hi=hi,
                                     n=int(len(a) + len(b)), excludes_zero=bool(lo > 0 or hi < 0)))
    # the table
    return pd.DataFrame(rows)

# stopwatch
t_tab = time.time()
# tables of all runs
SUMMARY, SIZEGRP, CONTRASTS = {}, {}, {}
# one folder per run
for run in RUNS:
    # this run's pages
    pp = per_page_full[per_page_full["run"] == run].copy()
    # its output folder
    rdir = os.path.join(EVAL_DIR, run)
    # create it
    os.makedirs(os.path.join(rdir, "figures"), exist_ok=True)
    # per_page.csv with exactly the contract columns
    pp[PER_PAGE_COLS].to_csv(os.path.join(rdir, "per_page.csv"), index=False)
    # summary and contrasts
    SUMMARY[run] = pd.concat([summarise_run(pp, run), image_auc_rows(pp, run)], ignore_index=True)
    # by size group
    SIZEGRP[run] = summarise_size_groups(pp, run)
    # contrasts
    CONTRASTS[run] = contrasts_run(pp, run)
    # save them
    SUMMARY[run].to_csv(os.path.join(rdir, "summary.csv"), index=False)
    # size-group summary
    SIZEGRP[run].to_csv(os.path.join(rdir, "summary_by_size_group.csv"), index=False)
    # contrasts
    CONTRASTS[run].to_csv(os.path.join(rdir, "contrasts.csv"), index=False)
    # one line per run
    print(run, "| summary rows", len(SUMMARY[run]), "| contrasts", len(CONTRASTS[run]), "| contrasts excluding 0:", int(CONTRASTS[run]["excludes_zero"].sum()) if len(CONTRASTS[run]) else 0)
# show the headline of every run: AI edits, PNG
for run in RUNS:
    # the summary
    s = SUMMARY[run]
    # headline rows
    h = s[(s["version"] == "ai_edit") & (s["format"] == "png") & (s["metric"].isin(["f1", "detection_rate", "pixel_auc"]))]
    # wide table: rows size x set, columns metric
    print("\n" + run + " (ai_edit, png):")
    # pivot
    print(h.pivot_table(index=["size", "set"], columns="metric", values="value").round(3).to_string())
# false-alarm headline
for run in RUNS:
    # untouched rows
    s = SUMMARY[run]
    # false alarm
    fa = s[s["metric"] == "false_alarm_rate"]
    # show
    print("\n" + run + " false-alarm rate (untouched pages):")
    # pivot
    print(fa.pivot_table(index=["size", "format"], columns="set", values="value").round(3).to_string())
print("\ntables done in", int(time.time() - t_tab), "s")

## 6. Figures per detector

**Why (ELI5):** the same five pictures for every detector, so they can be put side by side. Colours differ in
lightness (readable in greyscale); JPEG is hatched, PNG is solid; error bars are the 95% ranges.

(a) F1 by version × size, one panel per set. (b) detection rate, same layout. (c) false-alarm rate by size × format.
(d) forest plot of every F1 contrast with its 95% range (filled dot = range excludes 0). (e) F1 by size group.

In [ ]:
# grey level of each size (light to dark) so the figure still reads in greyscale
SIZE_GREY = {"shrink512": "0.85", "full": "0.55", "default": "0.25"}
# marker per contrast family (shapes differ, so colour is not needed)
FAMILY_MARKER = {"edit_type": "o", "resizing": "s", "compression": "^", "edit_size": "D", "edit_quality": "v", "training_data": "P"}

# make a value positive-safe for error bars
def yerr_of(vals, los, his):
    # lower and upper distances, NaN becomes 0
    lo = np.nan_to_num(np.array(vals, float) - np.array(los, float))
    # upper distance
    hi = np.nan_to_num(np.array(his, float) - np.array(vals, float))
    # never negative
    return np.vstack([np.clip(lo, 0, None), np.clip(hi, 0, None)])

# grouped bars: x = version, one bar per (size, format), one panel per set
def fig_version_bars(s, run, metric, ylabel, fname):
    # sizes this run has
    sizes = [z for z in SIZES_ALL if z in set(s["size"])]
    # one bar per size and format
    combos = [(z, f) for z in sizes for f in FORMATS]
    # three panels
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    # width of one bar
    w = 0.8 / max(len(combos), 1)
    # one panel per set
    for ax, set_ in zip(axes, SETS):
        # one bar group per combination
        for ci, (z, f) in enumerate(combos):
            # values, lows and highs over the four edited versions
            vals, los, his = [], [], []
            # each version
            for v in EDITED:
                # the matching summary row
                r = s[(s["size"] == z) & (s["set"] == set_) & (s["version"] == v) & (s["format"] == f) & (s["metric"] == metric)]
                # take the first row if there is one
                vals.append(float(r["value"].iloc[0]) if len(r) else np.nan)
                # low
                los.append(float(r["ci_lo"].iloc[0]) if len(r) else np.nan)
                # high
                his.append(float(r["ci_hi"].iloc[0]) if len(r) else np.nan)
            # x positions
            x = np.arange(len(EDITED)) + (ci - (len(combos) - 1) / 2) * w
            # bars: grey by size, hatched for JPEG
            ax.bar(x, np.nan_to_num(vals), w, color=SIZE_GREY[z], edgecolor="black", hatch="///" if f == "jpg" else "", label=z + " " + f,
                   yerr=yerr_of(vals, los, his), ecolor="black", capsize=1.5, error_kw=dict(lw=0.7))
        # labels
        ax.set_xticks(np.arange(len(EDITED)))
        # version names
        ax.set_xticklabels(EDITED, rotation=20)
        # panel title
        ax.set_title(set_)
        # y limit 0..1
        ax.set_ylim(0, 1.05)
    # y label on the first panel
    axes[0].set_ylabel(ylabel)
    # one legend below the panels
    handles, names = axes[0].get_legend_handles_labels()
    # draw it
    fig.legend(handles, names, loc="lower center", ncol=max(len(combos), 1), fontsize=8)
    # figure title
    fig.suptitle(run + ": " + ylabel + " (bars = mean over pages, whiskers = 95% range)")
    # tidy, leaving room for the legend
    fig.tight_layout(rect=[0, 0.08, 1, 0.95])
    # save
    fig.savefig(fname, dpi=130)
    # close it
    plt.close(fig)

# extra metrics: top row = share of an untouched page that is flagged, bottom row = image-level AUC (flagged-area score) per edited version
def fig_extra(s, run, fname):
    # sizes present
    sizes = [z for z in SIZES_ALL if z in set(s["size"])]
    # 2 rows x 3 sets
    fig, axes = plt.subplots(2, 3, figsize=(12, 6.6))
    # top row: area-based false alarm
    fa = s[s["metric"] == "false_alarm_area"]
    # each set
    for ax, set_ in zip(axes[0], SETS):
        # one bar per format
        for fi, f in enumerate(FORMATS):
            # values
            vals, los, his = [], [], []
            # each size
            for z in sizes:
                # the row
                r = fa[(fa["size"] == z) & (fa["set"] == set_) & (fa["format"] == f)]
                # numbers
                vals.append(float(r["value"].iloc[0]) if len(r) else np.nan)
                # low
                los.append(float(r["ci_lo"].iloc[0]) if len(r) else np.nan)
                # high
                his.append(float(r["ci_hi"].iloc[0]) if len(r) else np.nan)
            # positions
            x = np.arange(len(sizes)) + (fi - 0.5) * 0.4
            # bars
            ax.bar(x, np.nan_to_num(vals), 0.4, color="0.6", edgecolor="black", hatch="///" if f == "jpg" else "", label=f,
                   yerr=yerr_of(vals, los, his), ecolor="black", capsize=2)
        # ticks
        ax.set_xticks(np.arange(len(sizes)))
        # names
        ax.set_xticklabels(sizes)
        # title
        ax.set_title(set_)
        # label
        ax.set_ylabel("share of an untouched page flagged")
    # bottom row: image-level AUC with the flagged-area score
    au = s[s["metric"] == "image_auc_area"]
    # combinations of size and format
    combos = [(z, f) for z in sizes for f in FORMATS]
    # grey level per size
    shades = {z: str(0.85 - 0.3 * i) for i, z in enumerate(sizes)}
    # each set
    for ax, set_ in zip(axes[1], SETS):
        # each combination
        for ci, (z, f) in enumerate(combos):
            # values for the four edited versions
            vals, los, his = [], [], []
            # each version
            for v in EDITED:
                # the row
                r = au[(au["size"] == z) & (au["set"] == set_) & (au["format"] == f) & (au["version"] == v)]
                # numbers
                vals.append(float(r["value"].iloc[0]) if len(r) else np.nan)
                # low
                los.append(float(r["ci_lo"].iloc[0]) if len(r) else np.nan)
                # high
                his.append(float(r["ci_hi"].iloc[0]) if len(r) else np.nan)
            # bar width and positions
            w = 0.8 / len(combos)
            # positions
            x = np.arange(len(EDITED)) + (ci - (len(combos) - 1) / 2) * w
            # bars
            ax.bar(x, np.nan_to_num(vals), w, color=shades[z], edgecolor="black", hatch="///" if f == "jpg" else "", label=z + " " + f,
                   yerr=yerr_of(vals, los, his), ecolor="black", capsize=1.5)
        # chance line
        ax.axhline(0.5, color="black", lw=0.8, ls="--")
        # ticks
        ax.set_xticks(np.arange(len(EDITED)))
        # names
        ax.set_xticklabels(EDITED, fontsize=8)
        # limits
        ax.set_ylim(0, 1)
        # title
        ax.set_title(set_)
        # label
        ax.set_ylabel("image-level AUC (flagged-area score)")
    # legends
    axes[0][0].legend(fontsize=8)
    # second legend below the panels (so it never hides a bar)
    h2, l2 = axes[1][0].get_legend_handles_labels()
    # place it
    fig.legend(h2, l2, loc="lower center", ncol=len(l2), fontsize=8)
    # title
    fig.suptitle(run + ": area-based false alarm (top) and image-level AUC edited vs untouched (bottom); dashed = chance", fontsize=10)
    # layout
    fig.tight_layout(rect=[0, 0.05, 1, 0.95])
    # save
    fig.savefig(fname, dpi=130)
    # close
    plt.close(fig)

# false-alarm rate: x = size, bars = format, one panel per set
def fig_false_alarm(s, run, fname):
    # untouched rows
    fa = s[s["metric"] == "false_alarm_rate"]
    # sizes
    sizes = [z for z in SIZES_ALL if z in set(fa["size"])]
    # three panels
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)
    # one per set
    for ax, set_ in zip(axes, SETS):
        # one bar per format
        for fi, f in enumerate(FORMATS):
            # values
            vals, los, his = [], [], []
            # each size
            for z in sizes:
                # the row
                r = fa[(fa["size"] == z) & (fa["set"] == set_) & (fa["format"] == f)]
                # numbers
                vals.append(float(r["value"].iloc[0]) if len(r) else np.nan)
                # low
                los.append(float(r["ci_lo"].iloc[0]) if len(r) else np.nan)
                # high
                his.append(float(r["ci_hi"].iloc[0]) if len(r) else np.nan)
            # positions
            x = np.arange(len(sizes)) + (fi - 0.5) * 0.4
            # bars
            ax.bar(x, np.nan_to_num(vals), 0.4, color="0.6", edgecolor="black", hatch="///" if f == "jpg" else "", label=f,
                   yerr=yerr_of(vals, los, his), ecolor="black", capsize=2)
        # ticks
        ax.set_xticks(np.arange(len(sizes)))
        # names
        ax.set_xticklabels(sizes)
        # title
        ax.set_title(set_)
        # y limit
        ax.set_ylim(0, 1.05)
    # label
    axes[0].set_ylabel("false-alarm rate (untouched pages)")
    # legend
    axes[-1].legend()
    # title
    fig.suptitle(run + ": share of untouched pages with any pixel flagged (95% range)")
    # tidy
    fig.tight_layout()
    # save
    fig.savefig(fname, dpi=130)
    # close
    plt.close(fig)

# forest plot of contrasts (F1) for one table of contrasts
def fig_forest(con, title, fname, group_col=None):
    # only F1 contrasts
    c = con[con["metric"] == "f1"].copy()
    # nothing to draw
    if len(c) == 0:
        return False
    # a label for every row
    c["label"] = [" | ".join(x for x in row if x) for row in zip(c["contrast"], c["set"], c["size"], c["version"], c["format"])]
    # sort by family then label
    c = c.sort_values(["family", "label"]).reset_index(drop=True)
    # y positions (top to bottom)
    y = np.arange(len(c))[::-1]
    # tall figure for many rows
    fig, ax = plt.subplots(figsize=(9, max(3.5, 0.17 * len(c) + 1.5)))
    # a zero line
    ax.axvline(0, color="black", lw=1)
    # one marker shape per family
    for fam, g in c.groupby("family"):
        # rows of this family
        idx = g.index.values
        # the ranges
        ax.errorbar(g["diff"], y[idx], xerr=yerr_of(g["diff"], g["ci_lo"], g["ci_hi"]), fmt="none", ecolor="0.4", lw=0.8)
        # filled when the range excludes 0, hollow otherwise
        for i in idx:
            # the dot
            ax.scatter(c.loc[i, "diff"], y[i], marker=FAMILY_MARKER.get(fam, "o"), s=22, facecolors="black" if c.loc[i, "excludes_zero"] else "white",
                       edgecolors="black", zorder=3)
    # row labels
    ax.set_yticks(y)
    # text
    ax.set_yticklabels(c["label"], fontsize=5)
    # axis label
    ax.set_xlabel("difference in F1 (95% range); filled = range excludes 0")
    # title with the families
    ax.set_title(title + "  [" + ", ".join(sorted(c["family"].unique())) + "]", fontsize=9)
    # tidy
    fig.tight_layout()
    # save
    fig.savefig(fname, dpi=130)
    # close
    plt.close(fig)
    # drawn
    return True

# F1 by size group: one panel per set and size; bars = version; PNG only
def fig_size_groups(sg, run, fname):
    # F1, PNG only
    d = sg[(sg["metric"] == "f1") & (sg["format"] == "png")]
    # sizes
    sizes = [z for z in SIZES_ALL if z in set(d["size"])]
    # a grid: rows = sizes, columns = sets
    fig, axes = plt.subplots(len(sizes), 3, figsize=(13, 3.2 * len(sizes)), sharey=True, squeeze=False)
    # greys for the four versions
    vgrey = dict(zip(EDITED, ["0.15", "0.4", "0.65", "0.85"]))
    # each cell
    for ri, z in enumerate(sizes):
        # each set
        for ci, set_ in enumerate(SETS):
            # the axes
            ax = axes[ri][ci]
            # each version
            for vi, v in enumerate(EDITED):
                # values by group
                vals, los, his = [], [], []
                # small, medium, large
                for g in ["small", "medium", "large"]:
                    # row
                    r = d[(d["size"] == z) & (d["set"] == set_) & (d["version"] == v) & (d["size_group"] == g)]
                    # numbers
                    vals.append(float(r["value"].iloc[0]) if len(r) else np.nan)
                    # low
                    los.append(float(r["ci_lo"].iloc[0]) if len(r) else np.nan)
                    # high
                    his.append(float(r["ci_hi"].iloc[0]) if len(r) else np.nan)
                # positions
                x = np.arange(3) + (vi - 1.5) * 0.2
                # bars
                ax.bar(x, np.nan_to_num(vals), 0.2, color=vgrey[v], edgecolor="black", label=v, yerr=yerr_of(vals, los, his), ecolor="black", capsize=1.5)
            # ticks
            ax.set_xticks(range(3))
            # names
            ax.set_xticklabels(["small", "medium", "large"])
            # title
            ax.set_title(set_ + " | " + z, fontsize=9)
            # limit
            ax.set_ylim(0, 1.05)
    # label
    axes[0][0].set_ylabel("F1 (png)")
    # legend
    axes[0][-1].legend(fontsize=7)
    # title
    fig.suptitle(run + ": F1 by box-size group (PNG; 95% ranges)")
    # tidy
    fig.tight_layout()
    # save
    fig.savefig(fname, dpi=130)
    # close
    plt.close(fig)

# figure files per run
FIGS = {}
# make them
for run in RUNS:
    # the folder
    fdir = os.path.join(EVAL_DIR, run, "figures")
    # the files
    FIGS[run] = dict(a=os.path.join(fdir, "a_f1_by_version_size.png"), b=os.path.join(fdir, "b_detection_rate.png"),
                     c=os.path.join(fdir, "c_false_alarm.png"), d=os.path.join(fdir, "d_forest_contrasts.png"), e=os.path.join(fdir, "e_f1_by_size_group.png"),
                     f=os.path.join(fdir, "f_extra_metrics.png"))
    # (a) F1
    fig_version_bars(SUMMARY[run], run, "f1", "F1 vs the box mask at 0.5", FIGS[run]["a"])
    # (b) detection rate
    fig_version_bars(SUMMARY[run], run, "detection_rate", "detection rate", FIGS[run]["b"])
    # (c) false alarms
    fig_false_alarm(SUMMARY[run], run, FIGS[run]["c"])
    # (d) forest
    fig_forest(CONTRASTS[run], run + ": all F1 contrasts", FIGS[run]["d"])
    # (e) size groups
    fig_size_groups(SIZEGRP[run], run, FIGS[run]["e"])
    # (f) extra metrics: area-based false alarm and image-level AUC
    fig_extra(SUMMARY[run], run, FIGS[run]["f"])
    # report
    print(run, "figures:", sorted(os.listdir(os.path.join(EVAL_DIR, run, "figures"))))
# show the first run's figure (a) in the notebook
try:
    # display tool
    from IPython.display import Image as IPImage, display
    # show
    display(IPImage(filename=FIGS[next(iter(RUNS))]["a"]))
# not in a notebook
except Exception:
    # skip
    pass

## 7. Combined section (only when 2 or more detectors are present)

**Why (ELI5):** put the detectors next to each other on the SAME contrasts, and ask the training-data question: does a DAF
trained on one editor do better on that editor's pages than on the other editor's pages?

- **Seen vs unseen:** DAF weights `anytext` on AnyText pages (seen) versus DiffSTE pages (unseen), and the reverse. AnyText
  and DiffSTE share the same 49 pages, so the comparison is paired page by page.
- A missing detector or missing DAF weight is skipped with a printed note.

In [ ]:
# the folder for combined outputs
CDIR = os.path.join(EVAL_DIR, "combined")
# create it
os.makedirs(CDIR, exist_ok=True)
# the run ids we have
run_ids = list(RUNS)
# combined files that were written
COMBINED = {}
# only meaningful with 2+ runs
if len(run_ids) < 2:
    # explain
    print("Only", len(run_ids), "detector run found:", run_ids, "-> combined section skipped.")
# 2 or more runs
else:
    # one table: every summary row of every run
    all_sum = pd.concat([SUMMARY[r].assign(run=r) for r in run_ids], ignore_index=True)
    # save it
    all_sum.to_csv(os.path.join(CDIR, "summary_all_detectors.csv"), index=False)
    # one table: every contrast of every run
    all_con = pd.concat([CONTRASTS[r].assign(run=r) for r in run_ids], ignore_index=True)
    # save it
    all_con.to_csv(os.path.join(CDIR, "contrasts_all_detectors.csv"), index=False)
    # a compact comparison table: F1 and detection of AI edits (png), rows = set x size, columns = run
    comp = all_sum[(all_sum["version"] == "ai_edit") & (all_sum["format"] == "png") & (all_sum["metric"].isin(["f1", "detection_rate"]))].copy()
    # a text cell "value [lo, hi]"
    comp["cell"] = comp["value"].round(3).astype(str) + " [" + comp["ci_lo"].round(3).astype(str) + ", " + comp["ci_hi"].round(3).astype(str) + "]"
    # wide table
    comp_w = comp.pivot_table(index=["metric", "set", "size"], columns="run", values="cell", aggfunc="first")
    # save and show
    comp_w.to_csv(os.path.join(CDIR, "headline_comparison.csv"))
    # print
    print(comp_w.to_string())
    # the combined forest: same contrasts for every detector (AI edits, PNG where a format applies), F1 only
    cf = all_con[(all_con["metric"] == "f1") & (all_con["version"].isin(["", "ai_edit"])) & (all_con["format"].isin(["", "png"]))].copy()
    # a label per contrast
    cf["label"] = [" | ".join(x for x in row if x) for row in zip(cf["contrast"], cf["set"], cf["size"], cf["version"], cf["format"])]
    # draw one marker per detector run on each row
    if len(cf):
        # the rows (unique labels)
        labels = sorted(cf["label"].unique())
        # y positions
        ypos = {l: len(labels) - i for i, l in enumerate(labels)}
        # one marker shape per run
        marks = dict(zip(run_ids, ["o", "s", "^", "D", "v", "P", "X"]))
        # the figure
        fig, ax = plt.subplots(figsize=(9, max(3.5, 0.2 * len(labels) + 1.5)))
        # zero line
        ax.axvline(0, color="black", lw=1)
        # each run
        for ri, r in enumerate(run_ids):
            # its rows
            g = cf[cf["run"] == r]
            # small vertical shift so markers do not overlap
            yy = np.array([ypos[l] for l in g["label"]]) + (ri - (len(run_ids) - 1) / 2) * 0.2
            # whiskers
            ax.errorbar(g["diff"], yy, xerr=yerr_of(g["diff"], g["ci_lo"], g["ci_hi"]), fmt="none", ecolor="0.5", lw=0.7)
            # dots: filled when the range excludes 0
            ax.scatter(g["diff"], yy, marker=marks[r], s=22, facecolors=["black" if e else "white" for e in g["excludes_zero"]], edgecolors="black", label=r, zorder=3)
        # labels
        ax.set_yticks([ypos[l] for l in labels])
        # text
        ax.set_yticklabels(labels, fontsize=5)
        # axis
        ax.set_xlabel("difference in F1 (95% range); filled = range excludes 0")
        # title
        ax.set_title("All detectors on the same contrasts (AI edits, PNG)", fontsize=9)
        # legend
        ax.legend(fontsize=7)
        # tidy
        fig.tight_layout()
        # save
        COMBINED["forest"] = os.path.join(CDIR, "combined_forest.png")
        # write
        fig.savefig(COMBINED["forest"], dpi=130)
        # close
        plt.close(fig)
        # say so
        print("saved", COMBINED["forest"])

    # ---------- DAF weights: seen vs unseen editor ----------
    # the DAF runs that have weights named after an editor
    seen_rows = []
    # each weights name
    for wname in ["anytext", "diffste"]:
        # the run id
        rid = "daf_" + wname
        # missing: say so
        if rid not in RUNS:
            # note
            print("NOTE:", rid, "not found -> its seen/unseen rows are skipped")
            # next
            continue
        # per-page rows of this run: AI edits only
        pp = per_page_full[(per_page_full["run"] == rid) & (per_page_full["version"] == "ai_edit")]
        # the other editor
        other = "diffste" if wname == "anytext" else "anytext"
        # every size and format
        for (size, fmt), g in pp.groupby(["size", "format"]):
            # F1 and detection of the seen and the unseen editor's pages, paired by page
            for col, metric in [("f1", "f1"), ("detected", "detection_rate")]:
                # wide: page x set
                wide = g.pivot(index="page", columns="set", values=col)
                # both sets needed
                if wname not in wide.columns or other not in wide.columns:
                    continue
                # paired difference seen - unseen
                d = (wide[wname] - wide[other]).dropna()
                # mean and range
                m, lo, hi = boot_mean_ci(d.values)
                # seen and unseen means
                seen_rows.append(dict(weights=wname, size=size, format=fmt, metric=metric, seen_set=wname, unseen_set=other,
                                      seen_mean=float(wide[wname].mean()), unseen_mean=float(wide[other].mean()), diff_seen_minus_unseen=m,
                                      ci_lo=lo, ci_hi=hi, n=int(len(d)), excludes_zero=bool(lo > 0 or hi < 0)))
    # write the table when there is something
    if seen_rows:
        # table
        seen = pd.DataFrame(seen_rows)
        # save
        seen.to_csv(os.path.join(CDIR, "daf_seen_vs_unseen.csv"), index=False)
        # show F1 rows
        print(seen[seen["metric"] == "f1"].round(3).to_string(index=False))
        # remember
        COMBINED["seen"] = os.path.join(CDIR, "daf_seen_vs_unseen.csv")
    # none
    else:
        # note
        print("No DAF weights named anytext/diffste found -> seen/unseen table skipped.")

## 8. Hand-off pack (`handoff_pack.md`) and the 6 most important figures

**Why (ELI5):** one short file with only facts (numbers, counts, file names, no interpretation), plus six pictures, so
the results can be handed to someone (or to Claude) without sending the whole output. The nine sections follow
`prompt_handoff_pack.md` (1 run status, 2 main table PNG, 3 main table JPEG, 4 factor contrasts, 5 training data,
6 edit quality, 7 figures list, 8 settings and timings, 9 decisions). Every value is shown as `value [95% low, 95% high]`.

The file must stay under 400 lines. If it does not fit, the contrast and detector-comparison tables are shortened step by
step (first JPEG rows of the contrast tables, then sizes other than `shrink512`, then versions other than `ai_edit`, then the extra-metrics tables, a compact figure list and compact notebook 1 checks, then rows); the file always says what
was left out. Nothing is left out of the csv files.

In [ ]:
# ---------- small helpers ----------
# a pandas table as lines of a markdown table (at most max_rows rows)
def md_table(df, max_rows=60):
    # nothing to show
    if df is None or len(df) == 0:
        return ["(none)"]
    # copy so the original is not changed
    d = df.copy()
    # round float columns to 3 decimals
    for c in d.columns:
        # only numeric columns
        if pd.api.types.is_float_dtype(d[c]):
            # three decimals
            d[c] = d[c].round(3)
    # header
    lines = ["| " + " | ".join(str(c) for c in d.columns) + " |", "|" + "---|" * len(d.columns)]
    # rows
    for row in d.head(max_rows).itertuples(index=False):
        # one row
        lines.append("| " + " | ".join(str(v) for v in row) + " |")
    # say when rows were cut
    if len(d) > max_rows:
        # note
        lines.append("(" + str(len(d) - max_rows) + " more rows in the csv files)")
    # the lines
    return lines

# a value with its 95% range: 0.123 [0.100, 0.150]
def vr(v, lo, hi):
    # missing value
    if v is None or pd.isna(v):
        return "-"
    # three decimals each
    return "%.3f [%.3f, %.3f]" % (v, lo, hi)

# a readable detector name from the run id
def label(run):
    # fixed names
    names = {"adcd": "ADCD-Net", "dtd": "DTD"}
    # DAF weights
    if run.startswith("daf_"):
        return "DAF-" + run[4:]
    # the rest
    return names.get(run, run)

# every summary row of every run in one table (a column says which run)
def all_summary():
    # stack the runs
    return pd.concat([SUMMARY[r].assign(run=r) for r in RUNS], ignore_index=True)

# every contrast of every run in one table
def all_contrasts():
    # runs that have contrasts
    parts = [CONTRASTS[r].assign(run=r) for r in RUNS if len(CONTRASTS[r])]
    # nothing at all
    if not parts:
        return pd.DataFrame()
    # stack
    return pd.concat(parts, ignore_index=True)

# order of runs, sets and sizes for sorting
RUN_ORDER = {r: i for i, r in enumerate(RUNS)}
# the summary table once
SUM_ALL = all_summary()
# the contrast table once
CON_ALL = all_contrasts()
# the manifest with the image set and file stem (to find result files)
mf = manifest.copy()
# image set from the path images/<image set>/<version>/<format>/<file>
mf["img_set"] = mf["path"].str.split("/").str[1]
# page file stem
mf["stem"] = mf["path"].str.split("/").str[4].str.rsplit(".", n=1).str[0]
# the settings of the current attempt (the file is shortened step by step until it fits)
LEVEL = dict(cap=60, fmts=["png", "jpg"], sizes=None, versions=None, extra=["png", "jpg"], compact=False)

# ---------- section 1: run status ----------
def sec_status():
    # the lines
    lines = ["- date: " + datetime.now().strftime("%Y-%m-%d %H:%M") + " | SMOKE: " + str(SMOKE) + " | seed " + str(SEED) + " | bootstrap resamples " + str(N_BOOT)
             + " | flagged when map value >= " + str(THR_VALUE) + " of 255",
             "- dataset folder: " + os.path.basename(SETS_DIR.rstrip("/")) + " | manifest rows: " + str(len(manifest)),
             "- decisions.md attached: " + ("yes" if DECISIONS_TEXT else "no"),
             "- manifest fingerprint (md5, first 8 characters) of this dataset: " + MANIFEST_MD5[:8] + " | runs made from the same manifest (yes / no / unknown = not recorded): " + ", ".join(r["run"] + "=" + r["same_as_this_dataset"] for r in md5_rows), ""]
    # detectors not found
    for rid in EXPECTED_RUNS:
        # missing
        if rid not in RUNS:
            # line
            lines.append("- " + label(rid) + ": NOT FOUND (skipped)")
    # one row per run and size: result files found / expected, per set, for each format
    rows = []
    # every run
    for run, info in RUNS.items():
        # its sizes
        for size in sorted(set(EXPECTED_SIZES.get(info["detector"], SIZES_ALL)) | set(info["sizes"]), key=SIZES_ALL.index):
            # one row
            row = {"detector": label(run), "size": size}
            # each format
            for fmt in FORMATS:
                # text pieces and pass flag
                parts, ok = [], True
                # each set
                for s in SETS:
                    # manifest rows of this set and format
                    sub = mf[(mf["set"] == s) & (mf["format"] == fmt)]
                    # expected images and found result files
                    exp = len(sub)
                    found = sum(os.path.exists(result_path(run, size, r)) for r in sub.to_dict("records"))
                    # one piece
                    parts.append(str(found) + "/" + str(exp))
                    # pass only when complete
                    ok = ok and found == exp
                # the cell
                row[fmt + " images found/expected (" + "/".join(SETS) + ")"] = " ".join(parts) + (" PASS" if ok else " FAIL")
            # keep the row
            rows.append(row)
    # the table
    lines += ["", "Result maps found / expected (images = pages x versions listed in the manifest; a full run has 100 / 49 / 49 pages)"] + md_table(pd.DataFrame(rows), 40) + [""]
    # notebook 1 checks, if the file is in the dataset
    cpath = os.path.join(SETS_DIR, "checks.csv")
    # read it
    if os.path.exists(cpath):
        # the table
        ck = pd.read_csv(cpath, dtype=str, keep_default_na=False)
        # passed or not
        ok = [str(r.get("passed", "")).lower() == "true" for r in ck.to_dict("records")]
        # the short form: counts, and one line only for each failed check
        if LEVEL["compact"]:
            # counts
            lines.append("Notebook 1 checks (checks.csv): " + str(sum(ok)) + " PASS, " + str(len(ok) - sum(ok)) + " FAIL (full list in the dataset's checks.csv)")
            # failed ones
            for r, o in zip(ck.to_dict("records"), ok):
                # only failures
                if not o:
                    # line
                    lines.append("- FAIL | " + str(r.get("check", "")) + " | " + str(r.get("value", ""))[:90])
        # the long form: one line per check
        else:
            # heading
            lines.append("Notebook 1 checks (checks.csv)")
            # each check
            for r, o in zip(ck.to_dict("records"), ok):
                # pass or fail text
                lines.append("- " + ("PASS" if o else "FAIL") + " | " + str(r.get("check", "")) + " | " + str(r.get("value", ""))[:90])
    # no file
    else:
        # note
        lines.append("Notebook 1 checks: checks.csv not found in the dataset folder")
    # donor skips
    dpath = os.path.join(SETS_DIR, "donors.csv")
    # read it
    if os.path.exists(dpath):
        # the table
        dn = pd.read_csv(dpath, dtype=str, keep_default_na=False)
        # skipped flag as true / false
        dn["skipped_b"] = dn["skipped"].str.lower() == "true"
        # counts per set and version
        t = dn.groupby(["set", "version"]).agg(pages=("page", "count"), skipped=("skipped_b", "sum")).reset_index()
        # lines
        lines += ["", "Donor skips per set and version (donors.csv)"] + md_table(t, 20)
    # no file
    else:
        # note
        lines += ["", "Donor skips: donors.csv not found in the dataset folder"]
    # done
    return lines

# ---------- sections 2 and 3: main tables ----------
def main_table(set_name, fmt):
    # summary rows of this set and format (untouched rows too)
    s = SUM_ALL[(SUM_ALL["set"] == set_name) & (SUM_ALL["format"] == fmt)]
    # rows
    rows = []
    # every run and size in a fixed order
    for run in RUNS:
        # sizes
        for size in SIZES_ALL:
            # this run and size
            g = s[(s["run"] == run) & (s["size"] == size)]
            # nothing here
            if len(g) == 0:
                continue
            # the row starts with the names
            row = {"detector": label(run), "size": size}
            # each edited version: F1 and detection rate
            for v in EDITED:
                # both metrics
                for metric, short in [("f1", "F1"), ("detection_rate", "det")]:
                    # the row of the summary
                    q = g[(g["version"] == v) & (g["metric"] == metric)]
                    # value with range
                    row[short + " " + v] = vr(q["value"].iloc[0], q["ci_lo"].iloc[0], q["ci_hi"].iloc[0]) if len(q) else "-"
            # false-alarm rate on untouched pages
            q = g[(g["version"] == "untouched") & (g["metric"] == "false_alarm_rate")]
            # value with range
            row["false alarm (untouched)"] = vr(q["value"].iloc[0], q["ci_lo"].iloc[0], q["ci_hi"].iloc[0]) if len(q) else "-"
            # pages behind the numbers
            q1 = g[(g["version"] == "ai_edit") & (g["metric"] == "f1")]
            # n of AI-edit pages and of untouched pages
            row["n ai_edit"] = int(q1["n"].iloc[0]) if len(q1) else 0
            # untouched n
            row["n untouched"] = int(q["n"].iloc[0]) if len(q) else 0
            # keep
            rows.append(row)
    # the table, at most 15 rows
    return md_table(pd.DataFrame(rows), 15)

# extra metrics table: area-based false alarm and image-level AUC (AI edits versus untouched pages), one table per format
def extra_table(fmt):
    # summary rows of this format
    s = SUM_ALL[SUM_ALL["format"] == fmt]
    # rows
    rows = []
    # every run and size
    for run in RUNS:
        # sizes
        for size in SIZES_ALL:
            # this run and size
            g = s[(s["run"] == run) & (s["size"] == size)]
            # nothing
            if len(g) == 0:
                continue
            # the row
            row = {"detector": label(run), "size": size}
            # each set
            for set_ in SETS:
                # area-based false alarm on untouched pages
                q = g[(g["set"] == set_) & (g["version"] == "untouched") & (g["metric"] == "false_alarm_area")]
                # value with range
                row["untouched flagged area, " + set_] = vr(q["value"].iloc[0], q["ci_lo"].iloc[0], q["ci_hi"].iloc[0]) if len(q) else "-"
            # each set
            for set_ in SETS:
                # image-level AUC of the AI edits
                q = g[(g["set"] == set_) & (g["version"] == "ai_edit") & (g["metric"] == "image_auc_area")]
                # value with range
                row["image AUC ai_edit, " + set_] = vr(q["value"].iloc[0], q["ci_lo"].iloc[0], q["ci_hi"].iloc[0]) if len(q) else "-"
            # keep
            rows.append(row)
    # the table
    return md_table(pd.DataFrame(rows), 15)

# one section: three tables (one per set)
def sec_main(fmt):
    # lines
    lines = ["Mean F1 and detection rate (det) per version, and the false-alarm rate on untouched pages; format " + fmt + "; cells = value [95% low, 95% high]"]
    # one table per set
    for s in SETS:
        # heading and table
        lines += ["", "Set: " + s] + main_table(s, fmt)
    # the extra metrics (dropped first when the file has to be shortened)
    if fmt in LEVEL["extra"]:
        # heading and table
        lines += ["", "Extra metrics, format " + fmt + ": share of an untouched page that is flagged (area-based false alarm), and image-level AUC of the AI edits versus untouched pages (score = flagged-pixel share)"] + extra_table(fmt)
    # done
    return lines

# keep at most cap rows, shared evenly between the detectors (so no detector disappears)
def cut_per_detector(df, col):
    # no cut needed
    if len(df) <= LEVEL["cap"]:
        # whole table
        return md_table(df, 100000)
    # rows per detector
    per = max(2, LEVEL["cap"] // max(1, df[col].nunique()))
    # first rows of each detector
    cut = df.groupby(col, sort=False).head(per)
    # table plus a note of what is missing
    return md_table(cut, 100000) + ["(" + str(len(df) - len(cut)) + " more rows not shown, the same number of rows per detector; all rows are in the csv files)"]

# ---------- section 4: factor contrasts ----------
def contrast_table(family):
    # no contrasts at all
    if len(CON_ALL) == 0:
        return ["(none)"]
    # this family
    c = CON_ALL[CON_ALL["family"] == family]
    # formats: compression has no format value, keep those rows
    c = c[(c["format"] == "") | c["format"].isin(LEVEL["fmts"])]
    # sizes: keep rows without a size value
    if LEVEL["sizes"]:
        # filter
        c = c[(c["size"] == "") | c["size"].isin(LEVEL["sizes"])]
    # versions: keep rows without a version value
    if LEVEL["versions"]:
        # filter
        c = c[(c["version"] == "") | c["version"].isin(LEVEL["versions"])]
    # the columns that identify one row
    keys = ["run", "set", "contrast", "size", "version", "format"]
    # rows
    rows = []
    # one row per key
    for key, g in c.groupby(keys, sort=False):
        # names and values
        d = dict(zip(keys, key))
        # F1 and detection rate
        f = g[g["metric"] == "f1"]
        # detection
        t = g[g["metric"] == "detection_rate"]
        # the row
        d["F1 diff [95% range]"] = vr(f["diff"].iloc[0], f["ci_lo"].iloc[0], f["ci_hi"].iloc[0]) if len(f) else "-"
        # clear for F1
        d["clear? F1"] = ("yes" if bool(f["excludes_zero"].iloc[0]) else "no") if len(f) else "-"
        # detection
        d["det diff [95% range]"] = vr(t["diff"].iloc[0], t["ci_lo"].iloc[0], t["ci_hi"].iloc[0]) if len(t) else "-"
        # clear for detection
        d["clear? det"] = ("yes" if bool(t["excludes_zero"].iloc[0]) else "no") if len(t) else "-"
        # pages
        d["n"] = int(f["n"].iloc[0]) if len(f) else (int(t["n"].iloc[0]) if len(t) else 0)
        # keep
        rows.append(d)
    # nothing
    if not rows:
        return ["(none)"]
    # to a table
    df = pd.DataFrame(rows)
    # sort by run, set, contrast, size
    df = df.sort_values(by=["run", "set", "contrast", "size", "version", "format"], key=lambda col: col.map(RUN_ORDER) if col.name == "run" else col, kind="stable")
    # show the detector name
    df["run"] = [label(r) for r in df["run"]]
    # rename for readability
    df = df.rename(columns={"run": "detector"})
    # drop key columns that are empty everywhere
    for k in ["size", "version", "format"]:
        # empty column
        if (df[k] == "").all():
            # drop
            df = df.drop(columns=k)
    # a fair cut when rows must be dropped: the same number of rows for every detector
    return cut_per_detector(df, "detector")

# the five factors
FACTORS = [("edit_type", "Edit type: each DocTamper version minus ai_edit (paired by page)"),
           ("resizing", "Resizing: full minus shrink512, and default minus full for DAF (paired by page)"),
           ("compression", "Compression: jpg minus png (paired by page)"),
           ("edit_size", "Edit size: large minus small box group (unpaired)"),
           ("edit_quality", "Edit quality: OCR-correct minus not correct, AI edits only (unpaired)")]

# one section with five tables
def sec_contrasts():
    # what is shown
    shown = "formats " + "/".join(LEVEL["fmts"]) + ", sizes " + ("all" if not LEVEL["sizes"] else "/".join(LEVEL["sizes"])) + ", versions " + ("all" if not LEVEL["versions"] else "/".join(LEVEL["versions"]))
    # the lines
    lines = ["Differences (first condition minus second), cells = value [95% low, 95% high]; clear? = yes when the range excludes 0.",
             "Shown here: " + shown + ". Every row is in contrasts.csv in each detector folder."]
    # each factor
    for fam, title in FACTORS:
        # heading and table
        lines += ["", title] + contrast_table(fam)
    # done
    return lines

# ---------- section 5: training data ----------
def pair_table():
    # AI edits only
    pf = per_page_full[per_page_full["version"] == "ai_edit"]
    # the first group: ADCD-Net and DTD; the second: DAF runs
    first = [r for r in ["adcd", "dtd"] if r in RUNS]
    # DAF runs
    daf = [r for r in RUNS if r.startswith("daf")]
    # pairs: every ADCD-Net/DTD run against every DAF run, and ADCD-Net against DTD
    pairs = [(a, b) for a in first for b in daf] + ([("adcd", "dtd")] if len(first) == 2 else [])
    # rows
    rows = []
    # each pair
    for a, b in pairs:
        # sizes both detectors have
        for size in (LEVEL["sizes"] or ["shrink512", "full"]):
            # sets
            for s in SETS:
                # formats
                for fmt in LEVEL["fmts"]:
                    # the two detectors' rows by page
                    A = pf[(pf["run"] == a) & (pf["size"] == size) & (pf["set"] == s) & (pf["format"] == fmt)].set_index("page")
                    # second
                    B = pf[(pf["run"] == b) & (pf["size"] == size) & (pf["set"] == s) & (pf["format"] == fmt)].set_index("page")
                    # pages both have
                    common = A.index.intersection(B.index)
                    # nothing
                    if len(common) == 0:
                        continue
                    # the row
                    row = {"contrast": label(a) + " minus " + label(b), "set": s, "size": size, "format": fmt, "n": int(len(common))}
                    # F1 and detection
                    for col, short in [("f1", "F1"), ("detected", "det")]:
                        # paired differences
                        d = A.loc[common, col].astype(float).values - B.loc[common, col].astype(float).values
                        # mean and range
                        m, lo, hi = boot_mean_ci(d)
                        # cell
                        row[short + " diff [95% range]"] = vr(m, lo, hi)
                        # clear?
                        row["clear? " + short] = "yes" if (lo > 0 or hi < 0) else "no"
                    # keep
                    rows.append(row)
    # the table (a fair cut when rows must be dropped)
    return cut_per_detector(pd.DataFrame(rows), "contrast") if rows else ["(none)"]

# the section
def sec_training():
    # lines
    lines = []
    # seen / unseen table of the combined section
    if "seen" in COMBINED:
        # read
        t = pd.read_csv(COMBINED["seen"])
        # keep the formats shown and F1 and detection rate
        t = t[t["metric"].isin(["f1", "detection_rate"]) & t["format"].isin(LEVEL["fmts"])]
        # sizes
        if LEVEL["sizes"]:
            # filter
            t = t[t["size"].isin(LEVEL["sizes"])]
        # the difference with its range as one text cell
        t = t.assign(**{"diff seen minus unseen [95% range]": [vr(d, lo, hi) for d, lo, hi in zip(t["diff_seen_minus_unseen"], t["ci_lo"], t["ci_hi"])],
                        "clear?": ["yes" if (lo > 0 or hi < 0) else "no" for lo, hi in zip(t["ci_lo"], t["ci_hi"])]})
        # columns that exist
        cols = [c for c in ["weights", "size", "format", "metric", "seen_mean", "unseen_mean", "diff seen minus unseen [95% range]", "clear?", "n"] if c in t.columns]
        # lines
        lines += ["DAF weights seen vs unseen editor, AI edits, paired by page (AnyText-weights on anytext vs diffste pages, and the reverse); seen_mean and unseen_mean are means, diff = seen minus unseen"]
        # the table
        lines += md_table(t[cols], LEVEL["cap"])
    # nothing
    else:
        # note
        lines += ["(no DAF seen/unseen table: needs daf_anytext and/or daf_diffste results)"]
    # detectors compared on AI edits
    lines += ["", "ADCD-Net / DTD versus DAF, and ADCD-Net versus DTD, on ai_edit (paired by page; difference = first minus second)"]
    # the table
    lines += pair_table() if len(RUNS) >= 2 else ["(needs 2 or more detectors)"]
    # done
    return lines

# ---------- section 6: edit quality ----------
def sec_quality():
    # AI edit rows, one per set and page
    q = manifest[manifest["version"] == "ai_edit"].drop_duplicates(["set", "page"])
    # rows
    rows = []
    # each set
    for s in SETS:
        # this set
        g = q[q["set"] == s]
        # nothing
        if len(g) == 0:
            continue
        # OCR-correct as 0/1
        oc = g["ai_ocr_correct"].astype(float).values
        # NED as numbers
        ned = pd.to_numeric(g["ai_ned"], errors="coerce").values
        # share and range
        m1, l1, h1 = boot_mean_ci(oc)
        # NED mean and range
        m2, l2, h2 = boot_mean_ci(ned[~np.isnan(ned)])
        # the row
        rows.append({"set": s, "n pages": int(len(g)), "OCR-correct share": vr(m1, l1, h1), "mean NED (0 = exact)": vr(m2, l2, h2)})
    # lines
    return ["AI edits read by EasyOCR in the box: share read exactly right, and mean normalised edit distance (NED)"] + md_table(pd.DataFrame(rows), 10)

# ---------- section 7: figures list ----------
# what each figure file shows (from its file name and the figure titles in section 6 and 7)
FIG_TEXT = {"a_f1_by_version_size.png": "F1 by version and size, one panel per set, PNG solid and JPEG hatched",
            "b_detection_rate.png": "detection rate by version and size, one panel per set, PNG solid and JPEG hatched",
            "c_false_alarm.png": "false-alarm rate on untouched pages by size and format",
            "d_forest_contrasts.png": "forest plot of every F1 contrast with its 95% range and a zero line",
            "e_f1_by_size_group.png": "F1 by edit-size group (small / medium / large boxes)",
            "f_extra_metrics.png": "top: share of an untouched page that is flagged (area-based false alarm); bottom: image-level AUC, edited versus untouched pages",
            "combined_forest.png": "forest plot comparing the detectors on the same contrasts"}

# the figure files of this run, in a fixed order, as (name for the handoff folder, path)
ALL_FIGS = []
# each run's figures
for run in RUNS:
    # its figure folder
    fdir = os.path.join(EVAL_DIR, run, "figures")
    # each file
    for f in sorted(os.listdir(fdir)) if os.path.isdir(fdir) else []:
        # only pictures
        if f.endswith(".png"):
            # keep
            ALL_FIGS.append((run + "_" + f, os.path.join(fdir, f)))
# the combined figure
if "forest" in COMBINED:
    # keep
    ALL_FIGS.append(("combined_" + os.path.basename(COMBINED["forest"]), COMBINED["forest"]))

# the six most important: combined forest, F1 and false-alarm figure of the first detector, forest of each detector, then the rest
cands = []
# combined forest first
if "forest" in COMBINED:
    # add
    cands.append(("combined_forest", COMBINED["forest"]))
# the first detector: main F1 figure and false-alarm figure
first_run = next(iter(RUNS))
# add them
cands.append((first_run + "_f1_by_version_size", FIGS[first_run]["a"]))
# false alarm
cands.append((first_run + "_false_alarm", FIGS[first_run]["c"]))
# the forest of every detector
for run in RUNS:
    # add
    cands.append((run + "_forest_contrasts", FIGS[run]["d"]))
# then the main F1 and false-alarm figure of the other detectors
for key, name in [("a", "f1_by_version_size"), ("c", "false_alarm")]:
    # other runs
    for run in list(RUNS)[1:]:
        # add
        cands.append((run + "_" + name, FIGS[run][key]))
# the first six that exist
CHOSEN = []
# go in order
for name, path in cands:
    # six is enough
    if len(CHOSEN) >= 6:
        break
    # keep it if the file exists and is not already chosen
    if os.path.exists(path) and path not in [p for _, p in CHOSEN]:
        # add
        CHOSEN.append((name, path))

# the section
def sec_figures():
    # the chosen paths
    star = [p for _, p in CHOSEN]
    # the short form: what each file type shows once, then one line per detector
    if LEVEL["compact"]:
        # what each type shows
        lines = ["Every detector folder has figures/ with these files (what they show):"]
        # one line per file type
        for fname, text in FIG_TEXT.items():
            # skip the combined one here
            if fname != "combined_forest.png":
                # line
                lines.append("- " + fname + ": " + text)
        # one line per detector folder: its files and which of them are starred
        for run in RUNS:
            # files of this run
            fs = [os.path.basename(path) for name, path in ALL_FIGS if name.startswith(run + "_")]
            # starred ones
            st = [os.path.basename(path) for name, path in ALL_FIGS if name.startswith(run + "_") and path in star]
            # line
            lines.append("- " + run + "/figures/: " + str(len(fs)) + " files" + ("; starred: " + ", ".join(st) if st else ""))
        # the combined figure
        if "forest" in COMBINED:
            # line
            lines.append("- combined/combined_forest.png: " + FIG_TEXT["combined_forest.png"] + (" (starred)" if COMBINED["forest"] in star else ""))
        # done
        return lines
    # lines
    lines = ["Every figure file (star = one of the 6 copied to handoff_figures/); names are relative to eval_outputs/"]
    # each figure
    for name, path in ALL_FIGS:
        # relative path
        rel = os.path.relpath(path, EVAL_DIR)
        # description
        text = FIG_TEXT.get(os.path.basename(path), "figure")
        # line
        lines.append("- " + ("★ " if path in star else "") + rel + ": " + text)
    # done
    return lines

# ---------- section 8: settings and timings ----------
def sec_settings():
    # rows
    rows = []
    # each run
    for run, info in RUNS.items():
        # read run_info.json
        with open(os.path.join(info["folder"], "run_info.json")) as f:
            # dictionary
            ri = json.load(f)
        # the settings that matter: names containing these words
        keep = ["weight", "piece", "torch", "version", "size", "device", "gpu", "python"]
        # as text
        txt = "; ".join(k + "=" + str(v)[:80] for k, v in ri.items() if any(w in k.lower() for w in keep))
        # seconds per image from run_log.csv
        secs = "-"
        # the log file
        lpath = os.path.join(info["folder"], "run_log.csv")
        # read it
        if os.path.exists(lpath):
            # the table
            lg = pd.read_csv(lpath)
            # seconds per size
            if "seconds" in lg.columns and "size" in lg.columns:
                # mean per size
                m = lg.groupby("size")["seconds"].agg(["mean", "count"])
                # text
                secs = "; ".join(str(i) + " " + ("%.2f" % r["mean"]) + " s (n=" + str(int(r["count"])) + ")" for i, r in m.iterrows())
        # row
        rows.append({"detector": label(run), "run_info.json": txt if txt else "-", "mean seconds per image": secs})
    # the dataset config
    cpath = os.path.join(SETS_DIR, "config.json")
    # lines
    lines = md_table(pd.DataFrame(rows), 20)
    # notebook 1 config
    if os.path.exists(cpath):
        # read
        with open(cpath) as f:
            # text
            c1 = json.load(f)
        # line
        lines += ["", "Notebook 1 config.json: " + "; ".join(k + "=" + str(v)[:40] for k, v in c1.items())]
    # this notebook
    lines += ["", "Notebook 3 config.json: " + "; ".join(k + "=" + str(v) for k, v in config.items())]
    # done
    return lines

# ---------- section 9: decisions since 9 Nov ----------
def sec_decisions():
    # none attached
    if not DECISIONS_TEXT:
        # note
        return ["(decisions.md was not attached to this notebook; attach the notes dataset to include it)"]
    # lines to keep
    out = []
    # are we inside a kept entry?
    keep = False
    # each line
    for ln in DECISIONS_TEXT.splitlines():
        # a dated entry starts with - YYYY-MM-DD
        m = re.match(r"^\s*-\s*(\d{4}-\d{2}-\d{2})", ln)
        # a new entry
        if m:
            # keep when dated 2026-11-09 or later
            keep = m.group(1) >= "2026-11-09"
        # a heading ends the entry
        elif ln.startswith("#"):
            # stop keeping
            keep = False
        # copy the line unchanged
        if keep:
            # add
            out.append(ln)
    # nothing that new
    return out if out else ["(no entries dated 2026-11-09 or later)"]

# the nine sections in the order of prompt_handoff_pack.md
SECTIONS = [("1. Run status", sec_status), ("2. Main table, PNG", lambda: sec_main("png")), ("3. Main table, JPEG", lambda: sec_main("jpg")),
            ("4. Factor contrasts", sec_contrasts), ("5. Training data", sec_training), ("6. Edit quality per editor", sec_quality),
            ("7. Figures", sec_figures), ("8. Key settings and timings", sec_settings), ("9. Decisions since 9 Nov", sec_decisions)]

# the steps used to shorten the file (each step is a dictionary of settings); rows are only cut in the later steps
LADDER = [dict(cap=999, fmts=["png", "jpg"], sizes=None, versions=None, extra=["png", "jpg"], compact=False),
          dict(cap=999, fmts=["png"], sizes=None, versions=None, extra=["png"], compact=False),
          dict(cap=999, fmts=["png"], sizes=None, versions=["ai_edit"], extra=["png"], compact=False),
          dict(cap=999, fmts=["png"], sizes=["shrink512"], versions=["ai_edit"], extra=["png"], compact=False),
          dict(cap=40, fmts=["png"], sizes=["shrink512"], versions=["ai_edit"], extra=["png"], compact=True),
          dict(cap=25, fmts=["png"], sizes=["shrink512"], versions=["ai_edit"], extra=[], compact=True),
          dict(cap=15, fmts=["png"], sizes=["shrink512"], versions=["ai_edit"], extra=[], compact=True),
          dict(cap=8, fmts=["png"], sizes=["shrink512"], versions=["ai_edit"], extra=[], compact=True)]

# build the file
def build_handoff(max_lines=400):
    # try the longest version first, then shorter ones, until it fits
    for step in LADDER:
        # set the level
        LEVEL.update(step)
        # default table length for tables without their own limit
        md_table.__defaults__ = (step["cap"],)
        # the title
        lines = ["# Hand-off pack: Week 7 evaluation (numbers only, no interpretation)", ""]
        # each section
        for title, fn in SECTIONS:
            # heading and body
            lines += ["## " + title, ""] + fn() + [""]
        # it fits: stop trying
        if len(lines) < max_lines:
            # done
            break
    # still too long: cut and say so
    if len(lines) >= max_lines:
        # cut
        lines = lines[:max_lines - 2] + ["(cut at " + str(max_lines) + " lines; see the csv files)"]
    # the lines
    return lines

# write the pack
HANDOFF = build_handoff()
# path
HANDOFF_PATH = os.path.join(EVAL_DIR, "handoff_pack.md")
# write it
with open(HANDOFF_PATH, "w") as f:
    # lines
    f.write(chr(10).join(HANDOFF) + chr(10))
# report
print("handoff_pack.md:", len(HANDOFF), "lines | contrast level:", LEVEL)
# the pack must be shorter than 400 lines and must not have been cut
check("handoff_pack.md is under 400 lines", str(len(HANDOFF)), len(HANDOFF) < 400)
# the last line says "cut" when something was dropped
check("handoff_pack.md not cut", HANDOFF[-1][:5] != "(cut ", HANDOFF[-1][:5] != "(cut ", hard=False)

# ---------- copy the 6 most important figures ----------
# folder for the copies
HF_DIR = os.path.join(EVAL_DIR, "handoff_figures")
# create it
os.makedirs(HF_DIR, exist_ok=True)
# copy each chosen figure
for name, path in CHOSEN:
    # copy
    shutil.copy(path, os.path.join(HF_DIR, name + ".png"))
# report
print("handoff figures:", [n + ".png" for n, _ in CHOSEN])

## 9. Checks and the three zips

**Why (ELI5):** a last look that everything is where it should be, then three zips: (1) **all outputs**; (2) **safe for
Claude** (tables, numbers and charts only; this notebook never writes a page image, and the notes copy is left out);
(3) **needed next** (the hand-off pack, its figures and the headline tables).

In [ ]:
# the checks table
checks_df = pd.DataFrame(CHECKS)
# save it
checks_df.to_csv(os.path.join(EVAL_DIR, "checks.csv"), index=False)
# show it
print(checks_df[["check", "value", "passed", "hard"]].to_string(index=False, max_colwidth=100))
# stop if any hard check failed
stop_if_failed()

# make a zip from (file, name inside zip) pairs
def make_zip(zip_path, items):
    # write the zip
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        # each file
        for src, arc in items:
            # add it
            z.write(src, arc)
    # size in MB
    return os.path.getsize(zip_path) / 1e6

# every file under a folder as (path, name in zip)
def walk_items(folder, prefix, skip_names=()):
    # the list
    items = []
    # walk
    for root, dirs, fs in os.walk(folder):
        # each file
        for f in fs:
            # skip some
            if f in skip_names:
                continue
            # full path
            full = os.path.join(root, f)
            # name inside
            items.append((full, os.path.join(prefix, os.path.relpath(full, folder))))
    # done
    return items

# (1) everything
all_items = walk_items(EVAL_DIR, "eval_outputs")
# (2) safe for Claude: everything except the copy of decisions.md
safe_items = walk_items(EVAL_DIR, "eval_outputs_safe", skip_names=("decisions_copy.md",))
# (3) needed next: hand-off pack, its figures, the summary and contrast tables
next_items = [(HANDOFF_PATH, "eval_next/handoff_pack.md")] + walk_items(HF_DIR, "eval_next/handoff_figures")
# the per-run and combined tables
for root, dirs, fs in os.walk(EVAL_DIR):
    # tables only
    for f in fs:
        # summary and contrast tables
        if f in ("summary.csv", "contrasts.csv", "summary_all_detectors.csv", "contrasts_all_detectors.csv", "headline_comparison.csv", "daf_seen_vs_unseen.csv", "coverage.csv"):
            # full path
            full = os.path.join(root, f)
            # add
            next_items.append((full, os.path.join("eval_next", os.path.relpath(full, EVAL_DIR))))
# the zip files
ZIPS = {"1_ALL_outputs": os.path.join(WORKING, "w7_eval_1_ALL_outputs.zip"), "2_SAFE_for_claude": os.path.join(WORKING, "w7_eval_2_SAFE_for_claude.zip"),
        "3_NEXT_handoff": os.path.join(WORKING, "w7_eval_3_NEXT_handoff.zip")}
# build them
for key, items in [("1_ALL_outputs", all_items), ("2_SAFE_for_claude", safe_items), ("3_NEXT_handoff", next_items)]:
    # make and report
    mb = make_zip(ZIPS[key], items)
    # print
    print("%-20s %8.2f MB %5d files  %s" % (key, mb, len(items), ZIPS[key]))
# download links (relative to the working folder on Kaggle)
try:
    # IPython
    from IPython.display import FileLink, display
    # relative paths
    os.chdir(WORKING)
    # each zip
    for key, path in ZIPS.items():
        # label
        print(key)
        # link
        display(FileLink(os.path.relpath(path, WORKING)))
# not in a notebook
except Exception as e:
    # print paths
    print("no download links here:", e)
# reminders
if SMOKE:
    print("#" * 78)
    print("#  SMOKE=True: this was only a test. Set SMOKE=False before Save & Run All.")
    print("#" * 78)
print("DONE. Detectors evaluated:", list(RUNS))